### 사전 준비: 필요한 패키지 설치

In [ ]:
!pip install boto3 bedrock-agentcore strands-agents strands-agents-tools bedrock-agentcore-starter-toolkit mcp --upgrade --quiet

**!!!커널 재시작 필요!!!**

상단 Kernel > Restart kernel 버튼 클릭

### 데이터 준비

In [ ]:
!python3 generate_sample_cur.py --rows 100 -o test_cur.csv

In [ ]:
CUR_FILE_NAME = "test_cur.csv"

In [ ]:
#import os
#os.environ['AWS_PROFILE'] = '<Your Profile Name>'
# SageMaker 환경은 필요 없음

import boto3
from strands import Agent


region = boto3.session.Session().region_name
sts = boto3.client('sts')
response = sts.get_caller_identity()
account_id = response['Account']

print(region)
print(account_id)

### Code Interpreter를 이용하여 CUR Report 분석하는 Agent를 만들어봅시다.

#### 1. Naive Approach: Agent의 Input으로 유저 쿼리와 CUR Report를 전달

이 방식은 CUR 파일 전체를 메시지에 포함시켜 Agent에게 전달합니다.

![image.png](./images/bedrock_agentcore_code_interpreter_1.png)

**문제:**
- 대용량 파일의 경우 토큰 사용량이 매우 높음
- 응답 속도가 느림
- 계산이 포함된 경우 답이 틀릴 가능성이 높다.

In [ ]:
with open(CUR_FILE_NAME, 'rb') as f:
    document_bytes = f.read()

#### 에이전트 초기화

CUR(Cost and Usage Report) CSV 파일을 분석하는 AI 에이전트를 설정합니다.

- **시스템 프롬프트**: CUR 파일 분석 역할 정의
- **메시지 캐시**: CSV 문서를 대화 컨텍스트에 미리 로드

In [ ]:
# Define the agent's system prompt
SYSTEM_PROMPT = """
당신은 Cost And Usage Report 파일을 분석하는 AI 어시스턴트입니다.

사용자가  Cost And Usage Report 파일 파일에 대해 질문하면, 파일을 분석하여 사용자의 쿼리에 응답합니다.
"""

# Create a conversation, and add a messages cache point to cache the conversation up to that point
messages = [
    {
        "role": "user",
        "content": [
            {
                "document": {
                    "format": "csv",
                    "name": "cur_report",
                    "source": {
                        "bytes": document_bytes,
                    },
                },
            },  
            {
                "text": "Use this document in your response."
            },
        ],
    },
    {
        "role": "assistant",
        "content": [
            {
                "text": "I will reference that document in my following responses."
            }
        ]
    }
]


# Create an agent with the model and messages
agent = Agent(
    system_prompt=SYSTEM_PROMPT,
    messages=messages
)

In [ ]:
response = agent([
    {"text": "서비스별 비용 총합을 계산해줘"}    
])

#### Pandas를 이용해서 답이 맞는지 확인해봅시다.

In [ ]:
import pandas as pd
df = pd.read_csv(CUR_FILE_NAME)
df.groupby('lineItem/ProductCode').sum('lineItem/UnblendedCost')['lineItem/UnblendedCost']


#### 응답 메트릭 확인

에이전트 응답의 토큰 사용량과 지연 시간을 출력합니다.


In [ ]:
input_tokens_used = response.metrics.accumulated_usage['inputTokens']
output_tokens_used = response.metrics.accumulated_usage['outputTokens']
e2e_latency = response.metrics.accumulated_metrics['latencyMs']
# 토큰 사용량 및 지연 시간 출력
print(f"📊 Metrics")
print(f"├─ 입력 토큰: {input_tokens_used:,}")
print(f"├─ 출력 토큰: {output_tokens_used:,}")
print(f"├─ 총 토큰: {input_tokens_used + output_tokens_used:,}")
print(f"└─ 지연 시간: {e2e_latency:,}ms ({e2e_latency/1000:.2f}초)")


**결과 분석:**
- 입력 토큰: > 40,000 (매우 높음)
- 전체 CSV 파일이 메시지에 포함되어 토큰 사용량이 급증
- 계산 결과도 맞지 않음

#### 2. Code Interpreter를 tool로 등록하여 분석 코드를 작성하여 수행하게 함(Anti-pattern)

![image.png](./images/bedrock_agentcore_code_interpreter_2.png)


```python
from strands_tools.code_interpreter import AgentCoreCodeInterpreter


# Initialize the Code Interpreter tool
code_interpreter_tool = AgentCoreCodeInterpreter(region=region)

SYSTEM_PROMPT = """
당신은 AWS CUR(Cost and Usage Report) 분석 전문가입니다.

역할
사용자가 업로드한 CUR CSV 파일을 분석하여 비용 관련 질문에 답변합니다.

분석 수행
- 모든 분석은 executeCode 액션과 Python/pandas를 사용


응답 원칙
1. 항상 Code Interpreter로 실제 데이터 분석 수행
2. 구체적인 수치와 함께 답변
3. 한국어로 간결하게 응답
4. 질문에 직접 답변하고 불필요한 부연 설명 생략
5. 필요시 표 형식으로 결과 정리
"""

# Create a conversation, and add a messages cache point to cache the conversation up to that point
messages = [
    {
        "role": "user",
        "content": [
            {
                "document": {
                    "format": "csv",
                    "name": "cur_report",
                    "source": {
                        "bytes": document_bytes,
                    },
                },
            },  
            {
                "text": "Use this document in your response."
            },
        ],
    },
    {
        "role": "assistant",
        "content": [
            {
                "text": "I will reference that document in my following responses."
            }
        ]
    }
]


# Create an agent with the model and messages
agent = Agent(
    tools=[code_interpreter_tool.code_interpreter],
    system_prompt=SYSTEM_PROMPT,
    messages=messages
)

response = agent([
    {"text": "서비스별 비용 총합을 계산해줘"}
])
```

```python
input_tokens_used = response.metrics.accumulated_usage['inputTokens']
output_tokens_used = response.metrics.accumulated_usage['outputTokens']
e2e_latency = response.metrics.accumulated_metrics['latencyMs']
# 토큰 사용량 및 지연 시간 출력
print(f"📊 Metrics")
print(f"├─ 입력 토큰: {input_tokens_used:,}")
print(f"├─ 출력 토큰: {output_tokens_used:,}")
print(f"├─ 총 토큰: {input_tokens_used + output_tokens_used:,}")
print(f"└─ 지연 시간: {e2e_latency:,}ms ({e2e_latency/1000:.2f}초)")
```

이 방식도 여전히 CSV 파일 전체를 메시지에 포함시키면서 Code Interpreter를 도구로 사용합니다.

**문제점:**
- 정확한 결과를 출력하지만 코드 내에서 데이터를 거대한 string으로 저장하고 이를 사용함.
- Code Interpreter가 실제 파일에 접근하지 않고 메시지의 데이터를 사용
- 비효율적인 구조

#### 3. raw 파일을 메시지에 포함시키지 않고 Code Interpreter에 미리 올려두고 Agent가 사용하도록 개선

이제 더 효율적인 방식을 사용합니다:

![image.png](./images/bedrock_agentcore_code_interpreter_3.png)

**개선 사항:**
1. Code Interpreter 세션을 먼저 생성
2. CSV 파일을 Code Interpreter 환경에 직접 업로드
3. Agent는 파일 경로만 참조하여 분석 수행

**예상 효과:**
- 토큰 사용량 감소
- 응답 속도 개선

In [ ]:
bedrock_agentcore_client = boto3.client('bedrock-agentcore')
response = bedrock_agentcore_client.start_code_interpreter_session(
    codeInterpreterIdentifier='aws.codeinterpreter.v1'
)
session_id = response['sessionId']
session_id

In [ ]:
try:
    with open(CUR_FILE_NAME, 'r', encoding='utf-8') as data_file_content:
        data_file_content = data_file_content.read()
    #print(data_file_content)
except FileNotFoundError:
    print(f"Error: The file '{data_file}' was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
response = bedrock_agentcore_client.invoke_code_interpreter(
    codeInterpreterIdentifier='aws.codeinterpreter.v1',
    sessionId=session_id,
    name='writeFiles',
    arguments={
        'content': [
            {
                'path': CUR_FILE_NAME,
                'text': data_file_content,
            },
        ],
    }
)
response

In [ ]:
response = bedrock_agentcore_client.invoke_code_interpreter(
    codeInterpreterIdentifier='aws.codeinterpreter.v1',
    sessionId=session_id,
    name='listFiles',
    arguments={
    }
)
response

In [ ]:
for event in response["stream"]:
     result = event["result"]
result

#### Code Interpreter 실행 도구 정의

Agent가 Code Interpreter를 통해 Python 코드를 실행할 수 있도록 `execute_python` 도구를 정의합니다.  
`tool_context.invocation_state`에서 세션 ID를 가져와 앞서 생성한 Code Interpreter 세션을 재사용하도록 합니다.

In [ ]:
from strands import tool, ToolContext


@tool(context=True)
def execute_python(code: str, description: str, tool_context: ToolContext):
    """Execute Python code"""

    if description:
        code = f"# {description}\n{code}"

    # Print code to be executed
    print(f"\n Code: {code}")

    code_interpreter_session_id = tool_context.invocation_state.get('session_id')

    code_client = boto3.client('bedrock-agentcore')
    response = code_client.invoke_code_interpreter(
        codeInterpreterIdentifier='aws.codeinterpreter.v1',
        sessionId=code_interpreter_session_id,
        name='executeCode',
        arguments={
            'code': code,
            'language': 'python',
            'clearContext': False
        }
    )

    for event in response["stream"]:
        return json.dumps(event["result"])

#### CUR 분석 에이전트 시스템 프롬프트 정의

에이전트가 사전 업로드된 CUR 파일을 Code Interpreter 환경에서 직접 읽고 Python으로 분석하도록 안내합니다.  
파일이 이미 세션에 업로드되어 있으므로, 별도 첨부 없이 바로 분석을 수행할 수 있습니다.



In [ ]:
import json
from strands import Agent, tool
from bedrock_agentcore.tools.code_interpreter_client import code_session
import asyncio

#Define the detailed system prompt for the assistant
SYSTEM_PROMPT = """
당신은 AWS CUR(Cost and Usage Report) 분석 전문가입니다.

## 주요 기능
- CUR 파일 분석 (CSV 지원)

## 파일 처리 가이드

### 사전 업로드된 파일 확인
분석 요청을 받으면 먼저 Code Interpreter 환경에서 사용 가능한 파일을 확인하세요

### 파일을 찾지 못한 경우
파일이 존재하지 않거나 읽을 수 없는 경우, 다음과 같이 응답하세요:

"CUR 파일을 찾을 수 없습니다. 분석할 Cost and Usage Report 파일을 업로드해 주세요."

## 응답 원칙
1. 항상 Code Interpreter로 실제 데이터 분석 수행
2. 구체적인 수치와 함께 답변
3. 한국어로 간결하게 응답
4. 질문에 직접 답변하고 불필요한 부연 설명 생략
5. 필요시 표 형식으로 결과 정리
"""


In [ ]:
# Create an agent with the Code Interpreter tool
agent = Agent(
    tools=[execute_python],
    system_prompt=SYSTEM_PROMPT
)

In [ ]:
agent_response = agent(
    [{"text": f"서비스별 비용을 계산해줘. 대상 파일 : {CUR_FILE_NAME}"}],
    session_id=session_id,
)

In [ ]:
input_tokens_used = agent_response.metrics.accumulated_usage['inputTokens']
output_tokens_used = agent_response.metrics.accumulated_usage['outputTokens']
e2e_latency = agent_response.metrics.accumulated_metrics['latencyMs']
# 토큰 사용량 및 지연 시간 출력
print(f"📊 Metrics")
print(f"├─ 입력 토큰: {input_tokens_used:,}")
print(f"├─ 출력 토큰: {output_tokens_used:,}")
print(f"├─ 총 토큰: {input_tokens_used + output_tokens_used:,}")
print(f"└─ 지연 시간: {e2e_latency:,}ms ({e2e_latency/1000:.2f}초)")

**결과 비교:**
- 여전히 높은 토큰 사용량 (44,779개)
- 이유: Agent가 여러 번 탐색적 코드를 실행하면서 토큰 소비
- 다음 단계에서 Dynamic System Prompting으로 더 개선 가능

### Dynamic System Prompting

CSV 파일의 컬럼 정보를 미리 추출하여 System Prompt에 포함시킵니다.

**핵심 아이디어:**
- Agent가 파일 구조를 탐색하는 시간과 토큰 낭비 방지
- 필요한 메타데이터를 사전에 제공하여 바로 분석 시작
- 컬럼명, 데이터 타입 등을 명시적으로 알려줌

In [ ]:
import csv

def extract_columns(csv_file_path):    
    with open(csv_file_path, 'r', encoding='utf-8') as f:
        columns = next(csv.reader(f))
    
    return columns


In [ ]:
columns = extract_columns(CUR_FILE_NAME)

In [ ]:
SYSTEM_PROMPT = f"""
당신은 AWS CUR(Cost and Usage Report) 분석 전문가입니다.

## 주요 기능
- CUR 파일 분석 (CSV 지원)

## 응답 원칙
1. 항상 Code Interpreter로 실제 데이터 분석 수행
2. 구체적인 수치와 함께 답변
3. 한국어로 간결하게 응답
4. 질문에 직접 답변하고 불필요한 부연 설명 생략
5. 필요시 표 형식으로 결과 정리


You have access to the following files:
<file path='{CUR_FILE_NAME}' type='text/csv'>\
<metadata>
{{"columns": {columns}, "recommended_library": "pandas"}}
</metadata>
</file>
"""

In [ ]:
print(SYSTEM_PROMPT)

In [ ]:
# Create an agent with the Code Interpreter tool
agent = Agent(
    tools=[execute_python],
    system_prompt=SYSTEM_PROMPT
)

agent_response = agent(
    [{"text": f"서비스별 비용을 계산해줘. 대상 파일 : {CUR_FILE_NAME}"}],
    session_id=session_id,
)

In [ ]:
input_tokens_used = agent_response.metrics.accumulated_usage['inputTokens']
output_tokens_used = agent_response.metrics.accumulated_usage['outputTokens']
e2e_latency = agent_response.metrics.accumulated_metrics['latencyMs']
# 토큰 사용량 및 지연 시간 출력
print(f"📊 Metrics")
print(f"├─ 입력 토큰: {input_tokens_used:,}")
print(f"├─ 출력 토큰: {output_tokens_used:,}")
print(f"├─ 총 토큰: {input_tokens_used + output_tokens_used:,}")
print(f"└─ 지연 시간: {e2e_latency:,}ms ({e2e_latency/1000:.2f}초)")

### 고급 System Prompt 기법

추가 개선 사항:
1. **에러 핸들링 패턴 명시**: Agent가 안정적인 코드 작성
2. **금지 사항 명확화**: 불필요한 탐색 코드 방지
3. **컬럼명 대소문자 주의**: 정확한 컬럼명 사용 강제

In [ ]:
SYSTEM_PROMPT = f"""
당신은 AWS CUR(Cost and Usage Report) 분석 전문가입니다.

## 주요 기능
- CUR 파일 분석 (CSV 지원)



You have access to the following files:
<file path='{CUR_FILE_NAME}' type='text/csv'>\
<metadata>
{{"columns": {columns}, "recommended_library": "pandas"}}
</metadata>
</file>
⚠️ IMPORTANT: Column names are case-sensitive. Copy column names exactly as provided - do not change capitalization.

### 코드 작성 규칙
#### 필수: 에러 핸들링 패턴
python
try:
 # 분석 로직 작성
 ...

except FileNotFoundError:
 print("❌ 파일을 찾을 수 없습니다.")
except PermissionError:
 print("❌ 파일 접근 권한이 없습니다.")
...
except Exception as e:
 print(f"❌ 오류 발생: {{e}}")

#### 금지
- 컬럼/구조 확인 코드 (스키마는 이미 제공됨)
- try-except 없는 코드

### 응답 원칙
1. 항상 Code Interpreter로 실제 데이터 분석 수행
2. 구체적인 수치와 함께 답변
3. 한국어로 간결하게 응답
4. 질문에 직접 답변하고 불필요한 부연 설명 생략
5. 필요시 표 형식으로 결과 정리
"""

In [ ]:
print(SYSTEM_PROMPT)

In [ ]:
# Create an agent with the Code Interpreter tool
agent = Agent(
    tools=[execute_python],
    system_prompt=SYSTEM_PROMPT
)

agent_response = agent(
    [{"text": f"서비스별 비용을 계산해줘. 대상 파일 : {CUR_FILE_NAME}"}],
    session_id=session_id,
)

In [ ]:
input_tokens_used = agent_response.metrics.accumulated_usage['inputTokens']
output_tokens_used = agent_response.metrics.accumulated_usage['outputTokens']
e2e_latency = agent_response.metrics.accumulated_metrics['latencyMs']
# 토큰 사용량 및 지연 시간 출력
print(f"📊 Metrics")
print(f"├─ 입력 토큰: {input_tokens_used:,}")
print(f"├─ 출력 토큰: {output_tokens_used:,}")
print(f"├─ 총 토큰: {input_tokens_used + output_tokens_used:,}")
print(f"└─ 지연 시간: {e2e_latency:,}ms ({e2e_latency/1000:.2f}초)")

### Code Execution with MCP: 더 효율적인 Context 관리

앞서 CUR 리포트 분석을 통해 Code Interpreter의 다양한 활용 패턴을 살펴보았습니다. 이번에는 MCP 기반 Agent 환경에서 Code Interpreter를 활용하여, 도구 정의(tool definitions)와 중간 결과(intermediate results)로 인한 컨텍스트 윈도우 과다 소비 문제를 해결하는 방법을 알아보겠습니다.

| 문제 | 설명 |
|------|------|
| Tool definitions overload | 모든 도구 정의를 컨텍스트에 미리 로드하면서 토큰 소비 |
| Intermediate results | 도구 호출 결과가 매번 모델 컨텍스트를 통과하며 토큰 낭비 |

이 문제는 Code Interpreter 자체의 한계가 아니라, Agent가 도구를 사용하는 방식의 한계입니다. Code Interpreter를 활용하면 이 문제를 근본적으로 해결할 수 있습니다.

##### 핵심 아이디어: Agent가 도구를 호출하지 않고, 코드를 작성한다

[Code execution with MCP: Building more efficient agents](https://www.anthropic.com/engineering/code-execution-with-mcp) 블로그에서 제시하는 핵심 패턴은 다음과 같습니다:

**기존 방식** — Agent가 도구를 직접 호출하면, 모든 도구 정의와 결과가 컨텍스트를 거칩니다.

**개선 방식** — Agent가 코드를 작성하여 MCP 서버와 상호작용하면:

| 기법 | 효과 |
|------|------|
| Progressive Disclosure | 도구 정의를 파일시스템처럼 탐색하여 필요한 것만 온디맨드 로드 |
| Context Efficient Tool Results | 중간 결과를 코드 실행 환경에서 필터링·변환 후 모델에 전달 |

// 기존: Agent가 도구를 직접 호출 → 모든 정의 + 결과가 컨텍스트 소비
```python
agent.call_tool("google_docs_read", { id: "abc123" })
agent.call_tool("salesforce_update", { id: "00Q5f", data: result })
```
![image.png](./images/bedrock_agentcore_code_interpreter_4.png)



// 개선: Agent가 코드를 작성 → 실행 환경에서 처리, 컨텍스트는 최소화
```python
transcript = google_docs.read("meeting_notes")
salesforce.save("prospect", transcript)
```
![image.png](./images/bedrock_agentcore_code_interpreter_5.png)



MCP 서버의 도구들을 코드 API로 노출하면, Agent는 필요한 도구만 탐색하고 중간 결과를 코드 안에서 처리합니다. 컨텍스트 윈도우에는 최종 결과만 전달됩니다.

// 블로그 결과: 150,000 토큰 → 2,000 토큰 (98.7% 절감)
```
servers/
├── google-drive/
│   ├── getDocument.ts
│   └── index.ts
├── salesforce/
│   ├── updateRecord.ts
│   └── index.ts
└── ...
```


이제 이 패턴을 직접 구현해보겠습니다. MCP 서버를 구축하고, Custom Code Interpreter 환경에서 Agent가 코드를 통해 MCP 도구를 사용하도록 만들어 봅니다.

#### MCP 서버 구축 및 활용

이제 Custom MCP 서버를 구축하고 Agent와 연동하는 방법을 알아봅니다.

**MCP (Model Context Protocol)란?**
- AI 모델이 외부 도구와 통신하기 위한 표준 프로토콜
- 문서 관리, 데이터베이스 접근 등 다양한 기능을 제공
- Agent가 비동기적으로 도구를 호출할 수 있음

Cognito User Pool을 생성(또는 기존 풀 재사용)하고 Bearer Token을 발급받습니다. 이 토큰은 이후 Code Interpreter 환경에서 MCP 서버를 호출할 때 인증 헤더로 사용됩니다.

In [ ]:
from workshop_helper.utils import get_or_create_cognito_pool

access_token = get_or_create_cognito_pool(refresh_token=True)
print(f"Access token: {access_token['bearer_token']}")


`bedrock_agentcore_starter_toolkit`의 `Runtime`을 사용하여 MCP 서버를 AgentCore에 배포합니다. `mcp_server.py`를 엔트리포인트로 지정하고, Cognito JWT 인증을 설정하여 보안된 MCP 엔드포인트를 생성합니다.

In [ ]:
import boto3
from bedrock_agentcore_starter_toolkit import Runtime
from workshop_helper.utils import get_ssm_parameter, create_agentcore_runtime_execution_role

# Initialize the runtime toolkit
boto_session = boto3.session.Session()
region = boto_session.region_name

execution_role_arn = create_agentcore_runtime_execution_role()

agentcore_runtime = Runtime()

# Configure the deployment
response = agentcore_runtime.configure(
    entrypoint="mcp_server.py",
    execution_role=execution_role_arn,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="sample_mcp_server",
    protocol="MCP",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "allowedClients": [
                get_ssm_parameter("/krug/codeinterpreter/agentcore/client_id")
            ],
            "discoveryUrl": get_ssm_parameter(
                "/krug/codeinterpreter/agentcore/cognito_discovery_url"
            ),
        }
    },
    # Add custom header allowlist for Authorization and custom headers
    #request_header_configuration={
    #    "requestHeaderAllowlist": [
    #        "Authorization",  # Required for OAuth propogation
    #        "X-Amzn-Bedrock-AgentCore-Runtime-Custom-H1",  # Custom header
    #    ]
    #},
)

print("Configuration completed:", response)

설정이 완료되면 `launch()`로 MCP 서버를 AgentCore 런타임에 배포합니다. Docker 이미지 빌드 → ECR 푸시 → 런타임 생성 순서로 진행됩니다.

In [ ]:
agentcore_runtime.launch()

배포 상태를 확인합니다. `READY` 상태가 되면 MCP 서버가 정상적으로 실행 중인 것입니다.

In [ ]:
agent_arn = agentcore_runtime.status().agent['agentRuntimeArn'] if agentcore_runtime.status().agent['status'] == "READY" else None

In [ ]:
agent_arn

### Custom Code Interpreter 환경 구축

앞서 구축한 문서 관리 MCP 서버(`mcp_server.py`)와 통신하려면 **네트워크 접근이 가능한 Custom Code Interpreter**가 필요합니다.
AWS 관리형 Code Interpreter(`aws.codeinterpreter.v1`)는 외부 네트워크 접근이 제한되므로, Custom Code Interpreter를 사용합니다.

블로그에서 설명한 Code Execution with MCP 패턴을 구현하기 위해:
1. `PUBLIC` 네트워크 모드로 Code Interpreter 생성 → MCP 서버 호출 가능
2. MCP 클라이언트 코드(`client.py`)와 서버 래퍼(`server/docs/`)를 S3를 통해 동기화
3. Agent가 코드를 작성하면, Code Interpreter 내에서 MCP 도구를 직접 호출

이렇게 하면 **도구 정의가 컨텍스트 윈도우를 차지하지 않고**, Agent가 필요한 도구만 코드로 import하여 사용합니다.

이 섹션에서는 먼저 문서 관리 MCP 서버로 패턴을 검증한 뒤, 이후 섹션에서 동일한 패턴을 CUR 리포트 분석에 적용합니다.

Custom Code Interpreter를 `PUBLIC` 네트워크 모드로 생성합니다. 관리형(`aws.codeinterpreter.v1`)과 달리 외부 네트워크 접근이 가능하므로, Code Interpreter 내부에서 MCP 서버 엔드포인트를 직접 호출할 수 있습니다. `executionRoleArn`에는 S3 접근 권한이 포함된 IAM 역할을 지정합니다.

In [ ]:
import time
from workshop_helper.utils import create_code_interpreter_role

role_arn = create_code_interpreter_role()
time.sleep(30)

In [ ]:
import boto3
import json
import time
from workshop_helper.utils import create_code_artifacts_bucket


CP_ENDPOINT_URL = f"https://bedrock-agentcore-control.{region}.amazonaws.com"
DP_ENDPOINT_URL = f"https://bedrock-agentcore.{region}.amazonaws.com"

# Update the accountId to reflect the correct S3 path.
S3_BUCKET_NAME = create_code_artifacts_bucket(account_id, region)

bedrock_agentcore_control_client = boto3.client(
    'bedrock-agentcore-control'
)
bedrock_agentcore_client = boto3.client(
    'bedrock-agentcore'
)


unique_name = f"CustomCodeInterpreterEnv_{int(time.time())}"
create_response = bedrock_agentcore_control_client.create_code_interpreter(
    name=unique_name,
    description="Combined test code sandbox",
    executionRoleArn=role_arn,
    networkConfiguration={
        "networkMode": "PUBLIC"
    }
)
code_interpreter_id = create_response['codeInterpreterId']
print(f"Created custom interpreter ID: {code_interpreter_id}")

Custom Code Interpreter에서 세션을 시작합니다. 세션은 30분(1800초) 타임아웃으로 설정되며, 이 세션 내에서 파일 업로드, 코드 실행 등이 이루어집니다.

In [ ]:
session_response = bedrock_agentcore_client.start_code_interpreter_session(
    codeInterpreterIdentifier=code_interpreter_id,
    name="combined-test-session",
    sessionTimeoutSeconds=1800
)

In [ ]:
session_id = session_response['sessionId']
session_id

블로그 패턴의 핵심 구현입니다. `code_artifacts/` 폴더에 있는 MCP 클라이언트(`client.py`)와 서버 래퍼(`server/`)를 S3에 업로드합니다. 이 파일들이 Code Interpreter 환경의 파일시스템 역할을 하며, Agent가 `from server import salesforce`로 필요한 도구만 import하여 사용할 수 있게 됩니다.

In [ ]:
!cd code_artifacts && aws s3 sync . s3://{S3_BUCKET_NAME}/code_artifacts/src

S3에 업로드한 코드 파일들을 Code Interpreter 세션 내부로 동기화합니다. `executeCommand`를 사용하여 `aws s3 sync` 명령을 실행하면, Code Interpreter의 작업 디렉토리에 `client.py`, `server/` 등이 배치됩니다.

In [ ]:
command_to_execute = f"aws s3 sync s3://{S3_BUCKET_NAME}/code_artifacts/src ."
response = bedrock_agentcore_client.invoke_code_interpreter(
    codeInterpreterIdentifier=code_interpreter_id,
    sessionId=session_id,
    name="executeCommand",
    arguments={
        "command": command_to_execute
    }
)


S3 동기화 결과를 확인합니다. `exitCode: 0`이면 성공입니다.

In [ ]:
for event in response["stream"]:
    print(json.dumps(event["result"], default=str, indent=2))

Code Interpreter 환경의 파일 목록을 확인합니다. `client.py`, `server/` 디렉토리가 보이면 MCP 도구 코드가 정상적으로 배치된 것입니다.

In [ ]:
response = bedrock_agentcore_client.invoke_code_interpreter(
    codeInterpreterIdentifier=code_interpreter_id,
    sessionId=session_id,
    name='listFiles',
    arguments={
    }
)

In [ ]:
for event in response["stream"]:
    print(json.dumps(event["result"], default=str, indent=2))

Agent가 생성한 코드를 Code Interpreter에서 실행하기 위한 템플릿을 정의합니다. 이 템플릿은 Agent가 작성한 async 코드를 `async def main()` 안에 삽입하고, MCP 서버 인증(Cognito 토큰)과 환경변수 설정을 자동으로 처리합니다. Agent는 비즈니스 로직만 작성하면 됩니다.

In [ ]:
mcp_server_arn = agent_arn

CODE_TEMPLATE = '''
import asyncio
import sys
import os
sys.path.insert(0, ".")

from workshop_helper.utils import get_or_create_cognito_pool


os.environ["REGION"] = "{region}"
os.environ["AGENT_ARN"] = "{mcp_server_arn}"
access_token = get_or_create_cognito_pool(refresh_token=True)
os.environ["BEARER_TOKEN"] = access_token['bearer_token']

async def main():
{agent_code}

asyncio.run(main())
'''

CUR 분석에서 사용한 `execute_python` 도구를 Custom Code Interpreter용으로 재정의합니다. 핵심 차이점은:
- Agent가 작성한 코드를 `CODE_TEMPLATE`에 삽입하여 `generated.py` 파일로 저장
- `executeCommand`로 `python3 generated.py`를 실행
- 이를 통해 Agent의 코드가 MCP 서버 인증과 함께 실행됨

이 방식이 블로그에서 말하는 "Code Execution with MCP" 패턴의 구현체입니다.

In [ ]:
# state로부터 받아온 Code Interpreter Session을 통해 코드를 수행
import json
import boto3
import textwrap
import json
from strands import Agent, tool, ToolContext
from bedrock_agentcore.tools.code_interpreter_client import code_session

@tool(context=True)
def execute_python(code: str, description: str, tool_context: ToolContext):
    """Execute Python code"""

    if description:
        code = f"# {description}\n{code}"

    code_interpreter_id = tool_context.invocation_state.get('code_interpreter_id')
    code_interpreter_session_id = tool_context.invocation_state.get('session_id')
    bedrock_agentcore_client = boto3.client('bedrock-agentcore')
    
    indented_code = textwrap.indent(code, '    ') 
    # 에이전트 코드를 템플릿에 삽입
    generated_code = CODE_TEMPLATE.format(agent_code=indented_code, mcp_server_arn=mcp_server_arn, region=region)
    print(generated_code)
    response = bedrock_agentcore_client.invoke_code_interpreter(
        codeInterpreterIdentifier=code_interpreter_id,
        sessionId=code_interpreter_session_id,
        name='writeFiles',
        arguments={
            'content': [
                {
                    'path': 'generated.py',
                    'text': generated_code,
                },
            ],
        }
    )
    command_to_execute = "python3 generated.py"
    response = bedrock_agentcore_client.invoke_code_interpreter(
        codeInterpreterIdentifier=code_interpreter_id,
        sessionId=code_interpreter_session_id,
        name="executeCommand",
        arguments={
            "command": command_to_execute
        }
    )

    for event in response["stream"]:
        return json.dumps(event["result"])


MCP 도구를 코드 API로 제공하는 System Prompt를 정의합니다. 블로그의 Progressive Disclosure 패턴을 적용하여, Agent에게 도구의 함수 시그니처만 알려주고 실제 구현은 `from server import docs`로 import하도록 안내합니다. 이렇게 하면 도구 정의가 수백 토큰 수준으로 줄어듭니다.

여러 버전의 System Prompt를 시도하며 최적의 형태를 찾는 과정도 포함되어 있습니다.

In [ ]:
import sys
sys.path.insert(0, "code_artifacts")

import importlib
import inspect

# server 하위 모듈 목록
module_names = ["google_drive", "salesforce"]

tool_sections = []
for mod_name in module_names:
    mod = importlib.import_module(f"server.{mod_name}")
    exported = getattr(mod, "__all__", [])
    
    lines = []
    for func_name in exported:
        func = getattr(mod, func_name)
        sig = inspect.signature(func)
        # 파라미터만 추출 (return annotation 제외)
        params = ", ".join(str(p) for p in sig.parameters.values())
        ret = "" if sig.return_annotation is inspect.Parameter.empty else f" -> {inspect.formatannotation(sig.return_annotation)}"
        doc = (inspect.getdoc(func) or "").split("\n")[0]
        entry = f"  - await {mod_name}.{func_name}({params}){ret}"
        if doc:
            entry += f"  # {doc}"
        lines.append(entry)
    
    if lines:
        tool_sections.append(f"### `from server import {mod_name}`\n" + "\n".join(lines))

tool_listing = "\n\n".join(tool_sections)


tool_listing = "\n\n".join(tool_sections)

SYSTEM_PROMPT = f"""You have access to MCP tools organized by module.

Available modules and functions (all async, use await):

{tool_listing}

Write Python async code to accomplish the user's request.

IMPORTANT RULES:
1. Import only the modules you need: `from server import docs, google_drive, salesforce`
2. Write your code INSIDE an async function (the code will be wrapped in `async def main():`)
3. Use `await` for all tool calls
4. Do NOT use `asyncio.run()` - it's already handled
5. Do NOT write `await` at the top level - only inside functions

Example of CORRECT code:
```python
from server import google_drive

results = await google_drive.gdrive_search_documents("Acme")
for doc in results:
    detail = await google_drive.gdrive_get_document(doc["document_id"])
    print(detail)
```
Only output the code, no explanations."""


sys.path.remove("code_artifacts")

print(SYSTEM_PROMPT)


In [ ]:

# Create an agent with the Code Interpreter tool
agent = Agent(
    tools=[execute_python],
    system_prompt=SYSTEM_PROMPT
)

### MCP 서버와 Agent 연동

이제 Agent가 MCP 서버의 문서 관리 도구를 사용할 수 있도록 설정합니다.

**사용 가능한 도구:**
- `list_documents()`: 모든 문서 ID 조회
- `get_document(doc_id)`: 문서 내용 가져오기
- `read_doc_contents(doc_id)`: 문서 읽기
- `edit_document(doc_id, old_str, new_str)`: 문서 편집

Agent에게 MCP 서버의 문서를 조회하도록 요청합니다. Agent는 `from server import docs`로 도구를 import하고, `await docs.list_documents()`와 `await docs.get_document(doc_id)`를 호출하는 코드를 작성합니다. 이 코드는 Code Interpreter 내에서 실행되어 MCP 서버와 직접 통신합니다.

In [ ]:
document_id = "doc_meeting_0115"
opportunity_id = "OPP-2026-001"

prompt = f"""
Read the meeting notes from Google Drive (document ID: {document_id}),
extract key points and action items, then update the Salesforce opportunity
(record ID: {opportunity_id}) Notes field with a concise summary.
"""

In [ ]:
agent_response = agent(
    [{"text": prompt}],
    code_interpreter_id=code_interpreter_id,
    session_id=session_id,
)

Agent의 메시지 히스토리를 확인합니다. 도구 호출 과정(toolUse → toolResult → 최종 응답)이 기록되어 있어, Agent가 어떤 코드를 생성하고 어떤 결과를 받았는지 추적할 수 있습니다.

In [ ]:
agent.messages

Code Execution with MCP 패턴의 토큰 효율성을 확인합니다. CUR 분석의 Naive 방식(~45,000 토큰)과 비교하면, MCP 도구를 코드 API로 제공했을 때 입력 토큰이 ~2,000 수준으로 대폭 감소한 것을 확인할 수 있습니다. 블로그에서 언급한 98.7% 절감 효과와 유사한 결과입니다.

In [ ]:
input_tokens_used = agent_response.metrics.accumulated_usage['inputTokens']
output_tokens_used = agent_response.metrics.accumulated_usage['outputTokens']
e2e_latency = agent_response.metrics.accumulated_metrics['latencyMs']
# 토큰 사용량 및 지연 시간 출력
print(f"📊 Metrics")
print(f"├─ 입력 토큰: {input_tokens_used:,}")
print(f"├─ 출력 토큰: {output_tokens_used:,}")
print(f"├─ 총 토큰: {input_tokens_used + output_tokens_used:,}")
print(f"└─ 지연 시간: {e2e_latency:,}ms ({e2e_latency/1000:.2f}초)")

In [ ]:
print("Starting cleanup process...")

# 1. 세션 종료
try:
    print(f"Stopping session: {session_id}")
    bedrock_agentcore_client.stop_code_interpreter_session(
        codeInterpreterIdentifier=code_interpreter_id,
        sessionId=session_id
    )
    print("✓ Session stopped successfully")
except Exception as e:
    print(f"✗ Error stopping session: {e}")

# 2. Code Interpreter 삭제
try:
    print(f"Deleting code interpreter: {code_interpreter_id}")
    bedrock_agentcore_control_client.delete_code_interpreter(
        codeInterpreterId=code_interpreter_id
    )
    print("✓ Code interpreter deleted successfully")
except Exception as e:
    print(f"✗ Error deleting code interpreter: {e}")

print("Cleanup completed!")
